In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import awkward as ak
from pathlib import Path
import pickle
import os
import dask.dataframe as dd

import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)
from torch.utils.data import Dataset, DataLoader

import torch.nn as nn
import torch.nn.functional as F

2.10.0
0.22.1


In [2]:
# Caricamento dati

file_path = "data/ZJetsToQQ_13TeV-madgraphMLM-pythia8-NEVENT10000-RS54000001.parquet"
#file_path = "data/WJetsToQQ_13TeV-madgraphMLM-pythia8-NEVENT10000-RS45000001.parquet"
parameters = [
    # Jets
    "FullReco_JetAK4_PT",
    "FullReco_JetAK4_Eta",
    "FullReco_JetAK4_Phi",
    "FullReco_JetAK4_Mass",
    "FullReco_JetAK4_Charge",
    "FullReco_JetAK4_Constituents",

    #Contituents
    "FullReco_PFCand_PT",
    "FullReco_PFCand_Eta",
    "FullReco_PFCand_Phi",
    "FullReco_PFCand_Charge",
    "FullReco_PFCand_Mass",
    "FullReco_PFCand_fUniqueID",
    "FullReco_PFCand_PID"
]

ddf = dd.read_parquet(file_path, parameters)

In [3]:
import numpy as np
import pandas as pd
import random
import torch
from torch.utils.data import Dataset, DataLoader

def prepare_masked_constituent_dataset(ddf, min_jet_pt=50, max_constituents=60,
                                        mask_strategy='random', min_constituents=3,
                                        n_partitions=None):
    """
    Prepara dataset per la ricostruzione di costituenti mascherati.
    Gestisce correttamente la metadata inference di Dask.
    """
    
    def process_partition(partition):
        examples = []
        
        for _, row in partition.iterrows():
            # Estrai dati (sono già array numpy)
            jet_pt = row.get('FullReco_JetAK4_PT')
            jet_eta = row.get('FullReco_JetAK4_Eta')
            jet_phi = row.get('FullReco_JetAK4_Phi')
            jet_mass = row.get('FullReco_JetAK4_Mass')
            jet_charge = row.get('FullReco_JetAK4_Charge', 0)
            jet_constituents = row.get('FullReco_JetAK4_Constituents')
            
            pfcand_pt = row.get('FullReco_PFCand_PT')
            pfcand_eta = row.get('FullReco_PFCand_Eta')
            pfcand_phi = row.get('FullReco_PFCand_Phi')
            pfcand_mass = row.get('FullReco_PFCand_Mass')
            pfcand_charge = row.get('FullReco_PFCand_Charge', 0)
            pfcand_id = row.get('FullReco_PFCand_fUniqueID')
            pfcand_type = row.get('FullReco_PFCand_PID')
            
            if any(x is None for x in [jet_pt, pfcand_pt, pfcand_id]):
                continue
            
            # Funzione per convertire a array 1D
            def to_1d_array(data):
                if data is None:
                    return np.array([])
                if isinstance(data, np.ndarray):
                    if data.ndim == 0:
                        return np.array([float(data)]) if np.isscalar(data) else np.array([data])
                    return data.flatten()
                if isinstance(data, (list, tuple)):
                    return np.array(data).flatten()
                return np.array([data])
            
            # Converti tutti gli array
            jet_pt_arr = to_1d_array(jet_pt).astype(float)
            jet_eta_arr = to_1d_array(jet_eta).astype(float)
            jet_phi_arr = to_1d_array(jet_phi).astype(float)
            jet_mass_arr = to_1d_array(jet_mass).astype(float)
            jet_charge_arr = to_1d_array(jet_charge).astype(float)
            
            pfcand_pt_arr = to_1d_array(pfcand_pt).astype(float)
            pfcand_eta_arr = to_1d_array(pfcand_eta).astype(float)
            pfcand_phi_arr = to_1d_array(pfcand_phi).astype(float)
            pfcand_mass_arr = to_1d_array(pfcand_mass).astype(float)
            pfcand_charge_arr = to_1d_array(pfcand_charge).astype(float)
            pfcand_id_arr = to_1d_array(pfcand_id)
            pfcand_type_arr = to_1d_array(pfcand_type)
            
            # Dizionario per lookup rapido (ID -> indice)
            id_to_idx = {}
            for i, id_val in enumerate(pfcand_id_arr):
                try:
                    id_to_idx[int(id_val)] = i
                except (ValueError, TypeError):
                    continue
            
            # Processa ogni jet
            for jet_idx in range(len(jet_pt_arr)):
                if jet_pt_arr[jet_idx] < min_jet_pt:
                    continue
                
                if jet_idx >= len(jet_constituents):
                    continue
                
                constituents_ids = jet_constituents[jet_idx]
                if constituents_ids is None:
                    continue
                
                # Assicura che constituents_ids sia iterabile
                if isinstance(constituents_ids, (int, float, np.integer)):
                    constituents_ids = [constituents_ids]
                elif hasattr(constituents_ids, 'flatten'):
                    constituents_ids = constituents_ids.flatten()
                
                # Estrai features dei costituenti validi
                all_features = []
                all_pt = []
                all_types = []
                all_charges = []
                valid_constituent_indices = []  # Indici originali in constituents_ids
                
                for i, cid in enumerate(constituents_ids):
                    try:
                        cid_int = int(cid)
                    except (ValueError, TypeError):
                        continue
                    
                    if cid_int not in id_to_idx:
                        continue
                    
                    idx = id_to_idx[cid_int]
                    
                    if idx >= len(pfcand_type_arr):
                        continue
                    
                    particle_pid = int(pfcand_type_arr[idx])
                    
                    # ESCLUDI PID 22
                    if particle_pid == 22:
                        continue
                    
                    charge_val = float(pfcand_charge_arr[idx]) if idx < len(pfcand_charge_arr) else 0.0
                    
                    features = [
                        float(pfcand_pt_arr[idx]) / float(jet_pt_arr[jet_idx]),
                        float(pfcand_eta_arr[idx]) - float(jet_eta_arr[jet_idx]),
                        (float(pfcand_phi_arr[idx]) - float(jet_phi_arr[jet_idx]) + np.pi) % (2 * np.pi) - np.pi,
                        charge_val,
                        1 if particle_pid == 0 else 0,
                        1 if abs(particle_pid) == 45 else 0,
                        1 if abs(particle_pid) == 65 else 0,
                        1 if abs(particle_pid) == 92 else 0,
                        np.log(max(float(pfcand_mass_arr[idx]), 1e-8)),
                    ]
                    
                    all_features.append(features)
                    all_pt.append(float(pfcand_pt_arr[idx]))
                    all_types.append(particle_pid)
                    all_charges.append(charge_val)
                    valid_constituent_indices.append(i)  # Salva l'indice originale
                
                if len(all_features) < min_constituents:
                    continue
                
                # Seleziona costituente da mascherare
                if mask_strategy == 'highest_pt':
                    mask_pos = np.argmax(all_pt)
                elif mask_strategy == 'lowest_pt':
                    mask_pos = np.argmin(all_pt)
                else:
                    mask_pos = random.randint(0, len(all_features) - 1)
                
                # Ottieni l'ID e l'indice corretto per il costituente mascherato
                masked_constituent_orig_idx = valid_constituent_indices[mask_pos]
                masked_constituent_id = constituents_ids[masked_constituent_orig_idx]
                masked_idx_in_pfcand = id_to_idx[int(masked_constituent_id)]
                
                # Target features (usa l'indice corretto nell'array pfcand)
                target_features = np.array([
                    all_pt[mask_pos] / float(jet_pt_arr[jet_idx]),
                    float(pfcand_eta_arr[masked_idx_in_pfcand]),
                    float(pfcand_phi_arr[masked_idx_in_pfcand]),
                    all_charges[mask_pos],
                    1 if all_types[mask_pos] == 0 else 0,
                    1 if abs(all_types[mask_pos]) == 45 else 0,
                    1 if abs(all_types[mask_pos]) == 65 else 0,
                    1 if abs(all_types[mask_pos]) == 92 else 0,
                    np.log(max(float(pfcand_mass_arr[masked_idx_in_pfcand]), 1e-8)),
                ], dtype=np.float32)
                
                # Tipo particella
                if all_types[mask_pos] == 0:
                    particle_type_target = 0
                elif abs(all_types[mask_pos]) == 45:
                    particle_type_target = 1
                elif abs(all_types[mask_pos]) == 65:
                    particle_type_target = 2
                elif abs(all_types[mask_pos]) == 92:
                    particle_type_target = 3
                else:
                    particle_type_target = 4
                
                # Crea sequenza mascherata
                masked_features = []
                for i, feat in enumerate(all_features):
                    if i == mask_pos:
                        masked_features.append([0.0] * len(feat))
                    else:
                        masked_features.append(feat)
                
                # Padding/truncation
                if len(masked_features) > max_constituents:
                    if mask_pos >= max_constituents:
                        continue
                    masked_features = masked_features[:max_constituents]
                elif len(masked_features) < max_constituents:
                    padding = [[0.0] * len(all_features[0]) for _ in range(max_constituents - len(masked_features))]
                    masked_features.extend(padding)
                
                # Global features
                global_features = {
                    'jet_pt': float(jet_pt_arr[jet_idx]),
                    'jet_eta': float(jet_eta_arr[jet_idx]),
                    'jet_phi': float(jet_phi_arr[jet_idx]),
                    'jet_mass': float(jet_mass_arr[jet_idx]),
                    'jet_charge': float(jet_charge_arr[jet_idx]) if jet_idx < len(jet_charge_arr) else 0.0,
                    'n_constituents': len(all_features),
                    'n_constituents_after_mask': len(all_features) - 1,
                }
                
                examples.append({
                    'global_features': global_features,
                    'masked_sequence': np.array(masked_features, dtype=np.float32),
                    'mask_position': mask_pos,
                    'target_features': target_features,
                    'particle_type_target': particle_type_target,
                    'particle_pid': all_types[mask_pos],
                    'jet_pt': float(jet_pt_arr[jet_idx]),
                })
        
        # Se non ci sono esempi, restituisci DataFrame vuoto
        if not examples:
            return pd.DataFrame(columns=['global_features', 'masked_sequence', 'mask_position',
                                          'target_features', 'particle_type_target', 'particle_pid', 'jet_pt'])
        
        return pd.DataFrame(examples)
    
    # =========================================================
    # ESECUZIONE CON META ESPLICITA
    # =========================================================
    # Definisci la struttura del DataFrame di ritorno per il metadata
    meta_dict = {
        'global_features': 'object',
        'masked_sequence': 'object',
        'mask_position': 'int64',
        'target_features': 'object',
        'particle_type_target': 'int64',
        'particle_pid': 'int64',
        'jet_pt': 'float64',
    }
    meta = pd.DataFrame(columns=list(meta_dict.keys())).astype(meta_dict)
    
    # Applica la funzione
    if n_partitions is not None:
        partitions_to_process = ddf.partitions[:n_partitions]
        result = partitions_to_process.map_partitions(process_partition, meta=meta).compute()
    else:
        result = ddf.map_partitions(process_partition, meta=meta).compute()
    
    return result

In [4]:
def inspect_data_structure(ddf, n_samples=1):
    """
    Ispeziona la struttura dei dati per capire il formato.
    """
    sample = ddf.head(n_samples)#.compute()
    
    print("="*60)
    print("STRUTTURA DATI")
    print("="*60)
    
    for col in ['FullReco_JetAK4_PT', 'FullReco_JetAK4_Constituents', 
                'FullReco_PFCand_PID', 'FullReco_PFCand_fUniqueID']:
        if col in sample.columns:
            print(f"\n{col}:")
            val = sample[col].iloc[0]
            print(f"  Tipo: {type(val)}")
            print(f"  Contenuto: {val}")
            if isinstance(val, str):
                print(f"  Stringa di lunghezza: {len(val)}")
                # Prova a vedere i primi caratteri
                print(f"  Primi 200 caratteri: {val[:200]}")
            elif isinstance(val, (list, np.ndarray)):
                print(f"  Lunghezza lista: {len(val)}")
                if len(val) > 0:
                    print(f"  Primo elemento: {val[0]}")
                    print(f"  Tipo primo elemento: {type(val[0])}")
            print(f"  Dask dtype: {ddf[col].dtype}")
    
    return sample

# Ispeziona i dati
sample = inspect_data_structure(ddf)

STRUTTURA DATI

FullReco_JetAK4_PT:
  Tipo: <class 'numpy.ndarray'>
  Contenuto: [27.83 15.55]
  Lunghezza lista: 2
  Primo elemento: 27.828125
  Tipo primo elemento: <class 'numpy.float16'>
  Dask dtype: object

FullReco_JetAK4_Constituents:
  Tipo: <class 'numpy.ndarray'>
  Contenuto: [array([70158, 66702, 67247, 66692, 66694, 69923, 69912, 69981, 66700,
        69924, 69998, 66801, 67139, 70067, 69937, 70034, 69963, 69948,
        70179, 67126, 70180, 70004, 70081, 69997, 70111, 67328, 67333,
        70157, 69982, 69983, 67022, 70033, 70049, 70066, 70003, 67250,
        67245, 70110, 70140, 70139, 70048, 67241, 67015, 70054, 67134,
        67132, 70125, 67238, 67243, 67240], dtype=int32)
 array([69980, 66853, 67064, 70131, 69954, 70061, 70072, 70132, 70015,
        67185, 66625, 66947, 70087, 70118, 66143, 69770, 69898, 69769,
        70088, 66949, 70016, 69730, 66235, 66336, 69758, 66340, 66237,
        69790, 69929, 69811, 66239, 69834, 69905, 70060, 66628, 69809,
        66839, 6

In [5]:
def analyze_constituent_composition(ddf, min_jet_pt=50):
    """
    Analizza la composizione dei jet dopo l'esclusione delle particelle con PID 22.
    """
    
    def process_partition(partition):
        stats = {
            'total_constituents': 0,
            'excluded_pid22': 0,
            'photons': 0,
            'pions': 0,
            'kaons': 0,
            'protons': 0,
            'other': 0,
            'jets_processed': 0
        }
        
        for _, row in partition.iterrows():
            # Estrai dati (sono già array numpy)
            jet_pt = row['FullReco_JetAK4_PT']
            jet_constituents = row['FullReco_JetAK4_Constituents']
            pfcand_id = row['FullReco_PFCand_fUniqueID']
            pfcand_type = row['FullReco_PFCand_PID']
            
            if jet_pt is None or pfcand_id is None:
                continue
            
            # Assicurati che siano array numpy
            jet_pt = np.array(jet_pt)
            pfcand_id = np.array(pfcand_id)
            pfcand_type = np.array(pfcand_type)
            
            # Dizionario per lookup rapido
            id_to_idx = {int(id_val): i for i, id_val in enumerate(pfcand_id)}
            
            for jet_idx in range(len(jet_pt)):
                # Filtra per pt
                if jet_pt[jet_idx] < min_jet_pt:
                    continue
                
                stats['jets_processed'] += 1
                
                # Ottieni costituenti per questo jet
                constituents_ids = jet_constituents[jet_idx]
                
                # Assicurati che sia un array
                if not isinstance(constituents_ids, np.ndarray):
                    constituents_ids = np.array(constituents_ids)
                
                for cid in constituents_ids:
                    cid_int = int(cid)
                    if cid_int not in id_to_idx:
                        continue
                    
                    stats['total_constituents'] += 1
                    idx = id_to_idx[cid_int]
                    
                    pid = int(pfcand_type[idx])
                    
                    if pid == 22:
                        stats['excluded_pid22'] += 1
                    elif pid == 0:
                        stats['photons'] += 1
                    elif abs(pid) == 45:
                        stats['pions'] += 1
                    elif abs(pid) == 65:
                        stats['kaons'] += 1
                    elif abs(pid) == 92:
                        stats['protons'] += 1
                    else:
                        stats['other'] += 1
        
        return pd.DataFrame([stats])
    
    # Esegui l'analisi con meta esplicita
    result = ddf.map_partitions(
        process_partition, 
        meta=pd.DataFrame(columns=[
            'total_constituents', 'excluded_pid22', 'photons', 
            'pions', 'kaons', 'protons', 'other', 'jets_processed'
        ])
    ).compute()
    
    total_stats = result.sum()
    
    print("\n" + "="*60)
    print("ANALISI COMPOSIZIONE COSTITUENTI")
    print("="*60)
    print(f"Jet processati: {total_stats['jets_processed']:.0f}")
    print(f"\nTotale costituenti: {total_stats['total_constituents']:.0f}")
    
    if total_stats['total_constituents'] > 0:
        print(f"  - Esclusi (PID 22): {total_stats['excluded_pid22']:.0f} "
              f"({100*total_stats['excluded_pid22']/total_stats['total_constituents']:.1f}%)")
        
        print(f"\nCostituenti inclusi:")
        included = total_stats['total_constituents'] - total_stats['excluded_pid22']
        if included > 0:
            print(f"  - Fotoni (PID 0): {total_stats['photons']:.0f} "
                  f"({100*total_stats['photons']/included:.1f}%)")
            print(f"  - Pioni (PID ±45): {total_stats['pions']:.0f} "
                  f"({100*total_stats['pions']/included:.1f}%)")
            print(f"  - Kaoni (PID ±65): {total_stats['kaons']:.0f} "
                  f"({100*total_stats['kaons']/included:.1f}%)")
            print(f"  - Protoni (PID ±92): {total_stats['protons']:.0f} "
                  f"({100*total_stats['protons']/included:.1f}%)")
            print(f"  - Altri: {total_stats['other']:.0f} "
                  f"({100*total_stats['other']/included:.1f}%)")
    print("="*60)
    
    return total_stats

# Esegui l'analisi
stats = analyze_constituent_composition(ddf, min_jet_pt=50)


ANALISI COMPOSIZIONE COSTITUENTI
Jet processati: 6179

Totale costituenti: 287168
  - Esclusi (PID 22): 168659 (58.7%)

Costituenti inclusi:
  - Fotoni (PID 0): 74355 (62.7%)
  - Pioni (PID ±45): 35372 (29.8%)
  - Kaoni (PID ±65): 5572 (4.7%)
  - Protoni (PID ±92): 2801 (2.4%)
  - Altri: 409 (0.3%)


In [6]:
# Test con 1 partizione
print("Test con 1 partizione...")
result_df = prepare_masked_constituent_dataset(
    ddf, 
    min_jet_pt=50,  # Abbassato da 50 a 10
    max_constituents=60,
    mask_strategy='random',
    min_constituents=2,  # Abbassato da 3 a 2
    n_partitions=1
)

print(f"Numero esempi: {len(result_df)}")
print(f"Tipo: {type(result_df)}")

if len(result_df) > 0:
    # Ora result_df è un DataFrame, non una lista
    print("\nEsempio di dati:")
    first_row = result_df.iloc[0]
    print(f"  Global features: {first_row['global_features']}")
    print(f"  Masked sequence shape: {first_row['masked_sequence'].shape}")
    print(f"  Target features shape: {first_row['target_features'].shape}")
    print(f"  Particle type: {first_row['particle_type_target']}")
    print(f"  Jet PT: {first_row['jet_pt']}")
    
    # Crea dataset PyTorch (devi adattare MaskedConstituentDataset)
    # dataset = MaskedConstituentDataset(result_df, normalize_targets=True)

Test con 1 partizione...
Numero esempi: 6176
Tipo: <class 'pandas.core.frame.DataFrame'>

Esempio di dati:
  Global features: {'jet_pt': 55.59375, 'jet_eta': 3.50390625, 'jet_phi': -1.650390625, 'jet_mass': 12.9296875, 'jet_charge': 1.0, 'n_constituents': 14, 'n_constituents_after_mask': 13}
  Masked sequence shape: (60, 9)
  Target features shape: (9,)
  Particle type: 0
  Jet PT: 55.59375


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class MaskedConstituentDataset(Dataset):
    """
    Dataset per la ricostruzione di costituenti mascherati.
    Accetta sia DataFrame pandas che lista di dizionari.
    """
    
    def __init__(self, data, normalize_targets=True):
        """
        data: pandas.DataFrame o list di dizionari
        """
        self.normalize_targets = normalize_targets
        
        # Converte DataFrame in lista se necessario
        if isinstance(data, pd.DataFrame):
            self.data = []
            for _, row in data.iterrows():
                self.data.append({
                    'global_features': row['global_features'],
                    'masked_sequence': row['masked_sequence'],
                    'mask_position': row['mask_position'],
                    'target_features': row['target_features'],
                    'particle_type_target': row['particle_type_target'],
                    'particle_pid': row['particle_pid'],
                    'jet_pt': row['jet_pt'],
                })
        else:
            self.data = data
        
        # Calcola statistiche per normalizzazione
        if normalize_targets:
            self.feature_ranges = self._compute_feature_ranges()
    
    def _compute_feature_ranges(self):
        """Calcola media e std per le feature del target"""
        all_targets = np.array([d['target_features'] for d in self.data])
        
        ranges = {
            'mean': np.mean(all_targets, axis=0),
            'std': np.std(all_targets, axis=0),
        }
        
        # Non normalizzare carica (indice 3) e features binarie (indici 4-7)
        for i in [3, 4, 5, 6, 7]:
            ranges['std'][i] = 1.0
            ranges['mean'][i] = 0.0
        
        # Evita divisione per zero
        ranges['std'][ranges['std'] < 1e-6] = 1.0
        
        return ranges
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Global features (7 features)
        global_feats = torch.tensor([
            item['global_features']['jet_pt'],
            item['global_features']['jet_eta'],
            item['global_features']['jet_phi'],
            item['global_features']['jet_mass'],
            item['global_features']['jet_charge'],
            item['global_features']['n_constituents'],
            item['global_features']['n_constituents_after_mask'],
        ], dtype=torch.float32)
        
        # Sequenza mascherata (max_constituents, 9)
        masked_seq = torch.tensor(item['masked_sequence'], dtype=torch.float32)
        
        # Posizione del mask
        mask_pos = torch.tensor(item['mask_position'], dtype=torch.long)
        
        # Target features (9 features)
        target = torch.tensor(item['target_features'], dtype=torch.float32)
        
        # Tipo di particella
        particle_type = torch.tensor(item['particle_type_target'], dtype=torch.long)
        
        # Normalizzazione target
        if self.normalize_targets:
            target_normalized = target.clone()
            # Normalizza solo pt_rel (indice 0), eta, phi, log_mass (indice 8)
            for i in [0, 1, 2, 8]:
                if i < len(target_normalized):
                    target_normalized[i] = (target_normalized[i] - self.feature_ranges['mean'][i]) / self.feature_ranges['std'][i]
            target = target_normalized
        
        return {
            'global_features': global_feats,
            'masked_sequence': masked_seq,
            'mask_position': mask_pos,
            'target': target,
            'particle_type': particle_type,
            'jet_pt': torch.tensor(item['jet_pt'], dtype=torch.float32),
        }
    
    def denormalize_target(self, normalized_target):
        """Riporta il target alla scala originale"""
        if not self.normalize_targets:
            return normalized_target
        
        denormalized = normalized_target.clone()
        for i in [0, 1, 2, 8]:
            if i < len(denormalized):
                denormalized[i] = denormalized[i] * self.feature_ranges['std'][i] + self.feature_ranges['mean'][i]
        return denormalized

In [8]:
# Crea il dataset
dataset = MaskedConstituentDataset(result_df, normalize_targets=True)
print(f"Dimensione dataset: {len(dataset)}")

# Verifica le statistiche di normalizzazione
if dataset.normalize_targets:
    print(f"\nStatistiche di normalizzazione:")
    print(f"  PT mean: {dataset.feature_ranges['mean'][0]:.4f}, std: {dataset.feature_ranges['std'][0]:.4f}")
    print(f"  Eta mean: {dataset.feature_ranges['mean'][1]:.4f}, std: {dataset.feature_ranges['std'][1]:.4f}")
    print(f"  Phi mean: {dataset.feature_ranges['mean'][2]:.4f}, std: {dataset.feature_ranges['std'][2]:.4f}")
    print(f"  Log mass mean: {dataset.feature_ranges['mean'][8]:.4f}, std: {dataset.feature_ranges['std'][8]:.4f}")

# Crea dataloader
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

# Test un batch
print("\nTest batch:")
batch = next(iter(dataloader))
print(f"  Global features shape: {batch['global_features'].shape}")
print(f"  Masked sequence shape: {batch['masked_sequence'].shape}")
print(f"  Target shape: {batch['target'].shape}")
print(f"  Particle type shape: {batch['particle_type'].shape}")
print(f"  Jet PT shape: {batch['jet_pt'].shape}")

# Distribuzione dei tipi di particelle
particle_types = np.array([d['particle_type_target'] for d in dataset.data])
unique, counts = np.unique(particle_types, return_counts=True)
type_names = ['photon', 'pion', 'kaon', 'proton', 'other']
print(f"\nDistribuzione tipi di particelle nel target:")
for t, c in zip(unique, counts):
    print(f"  {type_names[t] if t < len(type_names) else t}: {c} ({100*c/len(particle_types):.1f}%)")

Dimensione dataset: 6176

Statistiche di normalizzazione:
  PT mean: 0.0583, std: 0.0908
  Eta mean: -0.0469, std: 1.8693
  Phi mean: 0.0082, std: 1.8231
  Log mass mean: -10.7528, std: 8.0136

Test batch:
  Global features shape: torch.Size([64, 7])
  Masked sequence shape: torch.Size([64, 60, 9])
  Target shape: torch.Size([64, 9])
  Particle type shape: torch.Size([64])
  Jet PT shape: torch.Size([64])

Distribuzione tipi di particelle nel target:
  photon: 3484 (56.4%)
  pion: 2153 (34.9%)
  kaon: 336 (5.4%)
  proton: 177 (2.9%)
  other: 26 (0.4%)


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import numpy as np

# =========================================================
# VERIFICA GPU
# =========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizzo dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# =========================================================
# DEFINIZIONE DEL MODELLO (con gestione GPU)
# =========================================================
class ConstituentReconstructor(nn.Module):
    def __init__(self, 
                 seq_hidden_dim=128,
                 global_hidden_dim=64,
                 target_dim=9,
                 n_layers=2,
                 dropout=0.1):
        super().__init__()
        
        self.target_dim = target_dim
        input_dim = 9
        
        # LSTM Encoder (bidirezionale)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=seq_hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True
        )
        lstm_out_dim = seq_hidden_dim * 2
        
        # Global features MLP
        self.global_mlp = nn.Sequential(
            nn.Linear(7, global_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(global_hidden_dim, global_hidden_dim),
            nn.ReLU(),
        )
        
        combined_dim = lstm_out_dim + global_hidden_dim
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(combined_dim, combined_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(combined_dim // 2, combined_dim // 4),
            nn.ReLU(),
        )
        
        # Output layers
        self.output_layer = nn.Linear(combined_dim // 4, target_dim)
        self.type_classifier = nn.Linear(combined_dim // 4, 5)
        self.charge_classifier = nn.Linear(combined_dim // 4, 3)
    
    def forward(self, global_features, masked_sequence):
        # LSTM
        lstm_out, (hidden, cell) = self.lstm(masked_sequence)
        
        # Prendi l'ultimo stato nascosto
        if self.lstm.bidirectional:
            seq_encoded = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            seq_encoded = hidden[-1]
        
        # Global features
        global_encoded = self.global_mlp(global_features)
        
        # Combina
        combined = torch.cat([seq_encoded, global_encoded], dim=1)
        
        # Decoder
        hidden_decoder = self.decoder(combined)
        
        return {
            'target_features': self.output_layer(hidden_decoder),
            'particle_type_logits': self.type_classifier(hidden_decoder),
            'charge_logits': self.charge_classifier(hidden_decoder),
        }
    
    def compute_loss(self, predictions, targets, particle_type_targets, charge_targets):
        # MSE per feature continue (pt_rel, eta, phi, log_mass)
        feature_indices = [0, 1, 2, 8]
        mse_loss = F.mse_loss(
            predictions['target_features'][:, feature_indices], 
            targets[:, feature_indices]
        )
        
        total_loss = mse_loss
        
        # CrossEntropy per tipo particella
        ce_loss = F.cross_entropy(predictions['particle_type_logits'], particle_type_targets)
        total_loss += 0.2 * ce_loss
        
        # CrossEntropy per carica
        charge_idx = charge_targets + 1
        charge_loss = F.cross_entropy(predictions['charge_logits'], charge_idx)
        total_loss += 0.3 * charge_loss
        
        return {
            'total_loss': total_loss,
            'mse_loss': mse_loss,
            'ce_loss': ce_loss,
            'charge_loss': charge_loss,
        }


# =========================================================
# FUNZIONE DI TRAINING CON GPU
# =========================================================
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, device='cuda'):
    """
    Training loop ottimizzato per GPU.
    """
    # Sposta modello su GPU
    model = model.to(device)
    
    # Ottimizzatore
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    
    # Scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # ========== TRAINING ==========
        model.train()
        train_loss = 0.0
        train_mse = 0.0
        train_ce = 0.0
        train_charge_acc = 0.0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for batch in pbar:
            # Sposta batch su GPU
            global_feats = batch['global_features'].to(device)
            masked_seq = batch['masked_sequence'].to(device)
            targets = batch['target'].to(device)
            particle_type = batch['particle_type'].to(device)
            charge_targets = batch['target'][:, 3].long().to(device)
            
            optimizer.zero_grad()
            predictions = model(global_feats, masked_seq)
            
            loss_dict = model.compute_loss(predictions, targets, particle_type, charge_targets)
            
            loss_dict['total_loss'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss_dict['total_loss'].item()
            train_mse += loss_dict['mse_loss'].item()
            train_ce += loss_dict['ce_loss'].item()
            
            # Accuracy carica
            charge_pred = torch.argmax(predictions['charge_logits'], dim=1)
            charge_acc = (charge_pred == charge_targets + 1).float().mean().item()
            train_charge_acc += charge_acc
            
            pbar.set_postfix({
                'loss': f'{loss_dict["total_loss"].item():.4f}',
                'mse': f'{loss_dict["mse_loss"].item():.4f}',
                'ce': f'{loss_dict["ce_loss"].item():.4f}',
                'chg_acc': f'{charge_acc:.3f}'
            })
        
        # Calcola medie training
        n_train = len(train_loader)
        train_loss /= n_train
        train_mse /= n_train
        train_ce /= n_train
        train_charge_acc /= n_train
        
        # ========== VALIDATION ==========
        model.eval()
        val_loss = 0.0
        val_mse = 0.0
        val_ce = 0.0
        val_charge_acc = 0.0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]'):
                global_feats = batch['global_features'].to(device)
                masked_seq = batch['masked_sequence'].to(device)
                targets = batch['target'].to(device)
                particle_type = batch['particle_type'].to(device)
                charge_targets = batch['target'][:, 3].long().to(device)
                
                predictions = model(global_feats, masked_seq)
                loss_dict = model.compute_loss(predictions, targets, particle_type, charge_targets)
                
                val_loss += loss_dict['total_loss'].item()
                val_mse += loss_dict['mse_loss'].item()
                val_ce += loss_dict['ce_loss'].item()
                
                charge_pred = torch.argmax(predictions['charge_logits'], dim=1)
                val_charge_acc += (charge_pred == charge_targets + 1).float().mean().item()
        
        n_val = len(val_loader)
        val_loss /= n_val
        val_mse /= n_val
        val_ce /= n_val
        val_charge_acc /= n_val
        
        # Scheduler
        scheduler.step(val_loss)
        
        # Stampa riepilogo
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"{'='*60}")
        print(f"Train - Loss: {train_loss:.6f} | MSE: {train_mse:.6f} | CE: {train_ce:.6f} | Charge Acc: {train_charge_acc:.4f}")
        print(f"Val   - Loss: {val_loss:.6f} | MSE: {val_mse:.6f} | CE: {val_ce:.6f} | Charge Acc: {val_charge_acc:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Salva miglior modello
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'train_loss': train_loss,
            }, 'best_reconstructor.pt')
            print(f"✓ Best model saved (val_loss: {val_loss:.6f})")
        
        # Cleanup GPU cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return model


# =========================================================
# FUNZIONE PER VISUALIZZARE L'USO DELLA GPU
# =========================================================
def print_gpu_memory():
    """Stampa l'utilizzo della memoria GPU."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        cached = torch.cuda.memory_reserved(0) / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Cached: {cached:.2f} GB")


# =========================================================
# MAIN: SETUP E TRAINING
# =========================================================
if __name__ == "__main__":
    
    # Verifica GPU
    print("="*60)
    print("CONFIGURAZIONE GPU")
    print("="*60)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Dispositivo: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memoria totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # =========================================================
    # PREPARAZIONE DATI
    # =========================================================
    print("\n" + "="*60)
    print("PREPARAZIONE DATI")
    print("="*60)
    
    # result_df è già stato creato dalla funzione prepare_masked_constituent_dataset
    print(f"Esempi totali: {len(result_df)}")
    
    # Crea dataset
    dataset = MaskedConstituentDataset(result_df, normalize_targets=True)
    
    # Split train/val
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    print(f"Train size: {len(train_dataset)}")
    print(f"Val size: {len(val_dataset)}")
    
    # DataLoader (pin_memory=True per trasferimento più veloce su GPU)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=64, 
        shuffle=True, 
        num_workers=4,  # Aumenta se hai più CPU
        pin_memory=True if torch.cuda.is_available() else False
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=64, 
        shuffle=False, 
        num_workers=4,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    # =========================================================
    # CREAZIONE MODELLO
    # =========================================================
    print("\n" + "="*60)
    print("CREAZIONE MODELLO")
    print("="*60)
    
    model = ConstituentReconstructor(
        seq_hidden_dim=128,
        global_hidden_dim=64,
        target_dim=9,
        n_layers=2,
        dropout=0.1
    )
    
    # Conta parametri
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parametri totali: {total_params:,}")
    print(f"Parametri addestrabili: {trainable_params:,}")
    
    # Mostra uso memoria GPU prima del training
    print_gpu_memory()
    
    # =========================================================
    # TRAINING
    # =========================================================
    print("\n" + "="*60)
    print("INIZIO TRAINING")
    print("="*60)
    
    # Training
    model = train_model(
        model, 
        train_loader, 
        val_loader, 
        epochs=30, 
        lr=1e-3, 
        device=device
    )
    
    print("\n" + "="*60)
    print("TRAINING COMPLETATO")
    print("="*60)
    print_gpu_memory()

Utilizzo dispositivo: cuda
GPU: Tesla T4
Memoria GPU: 15.6 GB
CONFIGURAZIONE GPU
Dispositivo: cuda
GPU: Tesla T4
Memoria totale: 15.6 GB

PREPARAZIONE DATI
Esempi totali: 6176
Train size: 4940
Val size: 1236

CREAZIONE MODELLO
Parametri totali: 607,889
Parametri addestrabili: 607,889
GPU Memory - Allocated: 0.00 GB, Cached: 0.00 GB

INIZIO TRAINING


Epoch 1/30 [Val]: 100%|██████████| 20/20 [00:00<00:00, 38.78it/s]



Epoch 1/30
Train - Loss: 1.496639 | MSE: 1.496639 | CE: 1.052805 | Charge Acc: 0.5497
Val   - Loss: 1.410919 | MSE: 1.410919 | CE: 0.958498 | Charge Acc: 0.5497
LR: 0.001000
✓ Best model saved (val_loss: 1.410919)


Epoch 2/30 [Val]: 100%|██████████| 20/20 [00:00<00:00, 36.59it/s]



Epoch 2/30
Train - Loss: 1.330157 | MSE: 1.330157 | CE: 0.928179 | Charge Acc: 0.5692
Val   - Loss: 1.195587 | MSE: 1.195587 | CE: 0.947185 | Charge Acc: 0.5459
LR: 0.001000
✓ Best model saved (val_loss: 1.195587)


...


Epoch 28/30
Train - Loss: 0.750326 | MSE: 0.750326 | CE: 0.772674 | Charge Acc: 0.6315
Val   - Loss: 0.940819 | MSE: 0.940819 | CE: 0.923414 | Charge Acc: 0.5534
LR: 0.000250


Epoch 29/30 [Val]: 100%|██████████| 20/20 [00:00<00:00, 37.79it/s]



Epoch 29/30
Train - Loss: 0.738392 | MSE: 0.738392 | CE: 0.767920 | Charge Acc: 0.6329
Val   - Loss: 0.938668 | MSE: 0.938668 | CE: 0.917808 | Charge Acc: 0.5534
LR: 0.000250


Epoch 30/30 [Val]: 100%|██████████| 20/20 [00:00<00:00, 40.62it/s]


Epoch 30/30
Train - Loss: 0.730226 | MSE: 0.730226 | CE: 0.762465 | Charge Acc: 0.6325
Val   - Loss: 0.954623 | MSE: 0.954623 | CE: 0.934360 | Charge Acc: 0.5487
LR: 0.000250

TRAINING COMPLETATO
GPU Memory - Allocated: 0.02 GB, Cached: 0.12 GB
